In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Window-Function-Demo'). \
    master('yarn'). \
    getOrCreate()

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Date-Time-Demo'). \
#     master('local[*]'). \
#     getOrCreate()

In [5]:
sc = spark.sparkContext

# pyspark.sql.functions.from_json

<img src="images/from_json.jpg" align = "left">



Ref - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_json.html

In PySpark, `from_json` is a powerful function used to transform a string column containing JSON data into a structured format (like a `StructType` or a `MapType`). This is essential when dealing with data from Kafka, NoSQL databases, or raw text files where nested data is packed into a single string.

# 1. How it Works

The function requires two primary arguments:

**The Column:** The string column containing the JSON.

**The Schema:** A definition of what the JSON looks like so Spark knows how to parse it.

## The Logic Flow

**1. Read:** Spark looks at the string.

**2. Schema Mapping:** It attempts to map the JSON keys to the fields defined in your schema.

**Output:** It returns a new column of type `Struct`, which you can then "flatten" to get individual columns.

# 2. Working Example

Let's say you have a DataFrame where one column is a JSON string representing a user's profile.

**Step 1:** Define the Schema

First, we use `StructType` to define the expected structure of the JSON.

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import from_json, col

# Define the schema of the JSON inside the string
json_schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True)
])

**Step 2:** Create the Raw Data

In [5]:
data = [
    (1, '{"name": "Alice", "age": 30, "city": "New York"}'),
    (2, '{"name": "Bob", "age": 25, "city": "Seattle"}')
]
df = spark.createDataFrame(data, ["id", "raw_json"])
df.show(truncate = False)

+---+------------------------------------------------+
|id |raw_json                                        |
+---+------------------------------------------------+
|1  |{"name": "Alice", "age": 30, "city": "New York"}|
|2  |{"name": "Bob", "age": 25, "city": "Seattle"}   |
+---+------------------------------------------------+



**Step 3:** Apply `from_json`

We apply the function and then use `.*` to expand the struct into individual columns.

In [6]:
# Convert the string to a Struct
parsed_df = df.withColumn("data", from_json(col("raw_json"), json_schema))
print(parsed_df.printSchema())

# Flatten the struct to see individual columns
final_df = parsed_df.select("id", "data.*")

final_df.show()

root
 |-- id: long (nullable = true)
 |-- raw_json: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)
 |    |-- city: string (nullable = true)

None
+---+-----+---+--------+
| id| name|age|    city|
+---+-----+---+--------+
|  1|Alice| 30|New York|
|  2|  Bob| 25| Seattle|
+---+-----+---+--------+



# 3. Important Considerations

- **Corrupt Records:** If the JSON string doesn't match the schema (e.g., a letter where an integer is expected), PySpark will return null for those fields by default.

- **Options:** You can pass a dictionary of options as a third argument. For example, {"allowBackslashEscapingAnyCharacter": "true"}.

- **Performance:** Parsing JSON is computationally expensive. If you are doing this at scale (billions of rows), ensure your schema is as tight as possible (avoid unnecessary fields).

In [8]:
df = spark.read.format('csv')\
.option('header', 'true')\
.option('inferschema', 'true')\
.load("file:///home/talentum/test-jupyter/DAWPPS/json-usecases/labs/data/flim.csv")

df.show()

+---+--------------------+
| id|               movie|
+---+--------------------+
|  1|{ 'Title': 'Avata...|
|  2|{ 'Title': 'I Am ...|
|  3|{ 'Title': '300',...|
|  4|{ 'Title': 'The A...|
|  5|{ 'Title': 'The W...|
|  6|{ 'Title': 'Inter...|
+---+--------------------+



In [21]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

movieSchema = StructType([
    StructField('Title', StringType(), True),
    StructField('Year', StringType(), True),
    StructField('Rated', StringType(), True),
    StructField('Released', StringType(), True),
    StructField('Genre', StringType(), True)
])

In [22]:
movieDf = df.withColumn('movie', F.from_json('movie', movieSchema))
movieDf.printSchema()
movieDf.show()

root
 |-- id: integer (nullable = true)
 |-- movie: struct (nullable = true)
 |    |-- Title: string (nullable = true)
 |    |-- Year: string (nullable = true)
 |    |-- Rated: string (nullable = true)
 |    |-- Released: string (nullable = true)
 |    |-- Genre: string (nullable = true)

+---+--------------------+
| id|               movie|
+---+--------------------+
|  1|[Avatar, 2009, PG...|
|  2|[I Am Legend, 200...|
|  3|[300, 2006, R, 09...|
|  4|[The Avengers, 20...|
|  5|[The Wolf of Wall...|
|  6|[Interstellar, 20...|
+---+--------------------+



In [24]:
movieDf.select('id', 'movie.Title', 'movie.Year', 'movie.Rated'\
               , 'movie.Released', 'movie.Genre').show(truncate = False)

+---+-----------------------+----+-----+-----------+--------------------------------+
|id |Title                  |Year|Rated|Released   |Genre                           |
+---+-----------------------+----+-----+-----------+--------------------------------+
|1  |Avatar                 |2009|PG-13|18 Dec 2009|["Action","Adventure","Fantasy"]|
|2  |I Am Legend            |2007|PG-13|14 Dec 2007|["Drama","Horror","Sci-Fi"]     |
|3  |300                    |2006|R    |09 Mar 2007|["Action","Drama","Fantasy"]    |
|4  |The Avengers           |2012|PG-13|04 May 2012|["Action","Sci-Fi","Thriller"]  |
|5  |The Wolf of Wall Street|2013|R    |25 Dec 2013|["Biography","Comedy","Crime"]  |
|6  |Interstellar           |2014|PG-13|07 Nov 2014|["Adventure","Drama","Sci-Fi"]  |
+---+-----------------------+----+-----+-----------+--------------------------------+

